# Limpieza de datos del LMS: exploración y justificación

Este notebook documenta **qué problemas tienen los Excel originales del LMS y por qué se limpian como se limpian**.
Es el respaldo de la sección *Limpieza y modelado de datos* del informe técnico.

Para cada fuente se sigue el mismo esquema:

1. **Problema:** se muestra en el dato original.
2. **Decisión:** qué se hace y por qué.
3. **Verificación:** se comprueba en la tabla limpia que genera `src/limpieza.py`.

| Fuente | Función de limpieza | Tabla resultante |
|---|---|---|
| Padrón de matriculados | `limpiar_matricula()` | `data/processed/matricula.parquet` |
| Reporte de participación | `limpiar_participacion()` | `data/processed/participacion.parquet` |
| Reporte de trazabilidad | `limpiar_trazabilidad()` | `data/processed/trazabilidad.parquet` |
| Calificaciones | `limpiar_calificaciones()` | `data/processed/calificaciones.parquet` |

### Cómo ejecutarlo

- **En la computadora:** abrir el notebook desde la carpeta `notebooks/` del repo, con los Excel en `data/raw/`.
- **En Google Colab:** ejecutar la celda siguiente y subir **dos archivos a la vez**: `Datos_LMS_Anonimizados.zip`
  y `src/limpieza.py`. Si ya están cargados, no vuelve a pedirlos.

In [1]:
# Solo en Google Colab: subir Datos_LMS_Anonimizados.zip y limpieza.py
import os
import sys

if "google.colab" in sys.modules:
    import shutil
    import zipfile
    from pathlib import Path
    from google.colab import files

    os.chdir("/content")
    raw = Path("/content/data/raw")
    script = Path("/content/src/limpieza.py")
    if not any(raw.glob("*/*.xlsx")) or not script.exists():
        for nombre in files.upload():  # elegir el .zip y limpieza.py
            if nombre.endswith(".zip"):
                raw.mkdir(parents=True, exist_ok=True)
                with zipfile.ZipFile(nombre) as z:
                    z.extractall(raw)
            elif nombre.endswith(".py"):
                script.parent.mkdir(parents=True, exist_ok=True)
                shutil.move(nombre, script)
    print("Excel cargados:", len(list(raw.rglob("*.xlsx"))), "| limpieza.py:", "sí" if script.exists() else "NO")
else:
    print("No es Colab: se usan data/raw/ y src/limpieza.py del repo")

No es Colab: se usan data/raw/ y src/limpieza.py del repo


In [2]:
import re
import sys
import unicodedata
from pathlib import Path

import pandas as pd
from IPython.display import display

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
from src import limpieza

RAW = RAIZ / "data" / "raw"
MATERIAS = limpieza.MATERIAS


def nfc(texto):
    return unicodedata.normalize("NFC", texto)


def carpeta(materia):
    return next(c for c in RAW.iterdir() if c.is_dir() and nfc(c.name) == nfc(materia))


ESTILO = [
    {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "15px"), ("font-weight", "bold"),
                                      ("text-align", "left"), ("padding", "6px 0 10px")]},
    {"selector": "th", "props": [("text-align", "left"), ("padding", "5px 12px")]},
    {"selector": "td", "props": [("padding", "5px 12px")]},
    {"selector": "tbody tr:nth-child(even)", "props": [("background-color", "rgba(127,127,127,0.08)")]},
]


def tabla(df, titulo, pct=(), dec=1, indice=True):
    # escape="html": muestra el texto tal cual (por ejemplo "&#128226;" no se convierte en emoji)
    est = df.style.set_caption(titulo).format(precision=dec, decimal=",", thousands=".", na_rep="—", escape="html")
    if pct:
        est = est.format("{:.1f} %", subset=list(pct), decimal=",", thousands=".", na_rep="—", escape="html")
    numericas = list(df.select_dtypes("number").columns)
    texto = [c for c in df.columns if c not in numericas]
    if numericas:
        est = est.set_properties(subset=numericas, **{"text-align": "right"})
    if texto:
        est = est.set_properties(subset=texto, **{"text-align": "left"})
    if not indice:
        est = est.hide(axis="index")
    display(est.set_table_styles(ESTILO))


def chequeos(filas, titulo):
    # Tabla de controles: qué se revisó, qué dio y qué se decide
    df = pd.DataFrame(filas, columns=["Qué se revisó", "Resultado", "Qué hacemos"])
    df["Resultado"] = df["Resultado"].astype(str)
    tabla(df, titulo, indice=False)


def verificacion(filas, titulo):
    # Tabla de verificación sobre los datos limpios
    df = pd.DataFrame(filas, columns=["Verificación", "Esperado", "Obtenido"])
    df["Esperado"] = df["Esperado"].astype(str)
    df["Obtenido"] = df["Obtenido"].astype(str)
    df["¿OK?"] = ["✔" if e == o else "✘" for e, o in zip(df["Esperado"], df["Obtenido"])]
    tabla(df, titulo, indice=False)

## 1. Archivos de origen

Hay una carpeta por materia (calificaciones, participación y trazabilidad) y una carpeta `Matriculados` con el
padrón oficial. El `.zip` contiene exactamente los mismos archivos, así que no se usa.

In [3]:
NOMBRES_TIPO = {"calificaciones_alumnos": "Calificaciones", "reporte_participacion": "Participación",
                "reporte_trazabilidad": "Trazabilidad", "matriculados": "Padrón de matriculados"}
inventario = pd.DataFrame([
    {"Carpeta": nfc(f.parent.name),
     "Tipo": NOMBRES_TIPO["matriculados" if f.parent.name == "Matriculados" else f.name.rsplit("_", 2)[0]]}
    for f in RAW.rglob("*.xlsx")
])
conteo = pd.crosstab(inventario["Carpeta"], inventario["Tipo"])[list(NOMBRES_TIPO.values())]
conteo.index.name = None
conteo.columns.name = None
tabla(conteo, "Cantidad de archivos Excel por carpeta")

,Calificaciones,Participación,Trazabilidad,Padrón de matriculados
Algoritmos y Estructuras de Datos,5,1,1,0
Matriculados,0,0,0,3
Probabilidad y Estadística,9,1,1,0
Álgebra,3,1,1,0


**Problema común a todos los archivos:** ninguno empieza con el encabezado en la primera fila. Arriba hay
título, filtros aplicados y fecha de exportación. Por ejemplo, un archivo de calificaciones:

In [4]:
archivo = sorted(carpeta("Álgebra").glob("calificaciones_alumnos_*.xlsx"))[0]
crudo = pd.read_excel(archivo, header=None, nrows=10).iloc[:, :5].fillna("")
crudo.index = [f"fila {i + 1}" for i in crudo.index]
crudo.columns = ["A", "B", "C", "D", "E"]
tabla(crudo, f"Archivo original: {archivo.name}")

tabla(pd.DataFrame({
    "Archivo": ["Padrón de matriculados", "Participación", "Trazabilidad", "Calificaciones"],
    "Fila del encabezado": ["2", "6 y 7 (encabezado doble)", "6", "7"],
    "Información útil arriba del encabezado": ["Código y nombre de la materia", "Fecha de exportación y filtro de fechas",
                                               "Fecha de exportación y filtro de fechas",
                                               "Fecha de exportación y nombre de la evaluación"],
}), "Dónde empiezan los datos en cada tipo de archivo", indice=False)

,A,B,C,D,E
fila 1,Fecha de exportación: 24/10/2025 11:49,,,,
fila 2,Nombre del aula: TUCD-MD-Álgebra 2025,,,,
fila 3,Calificación: AUTOEVALUACION1,,,,
fila 4,Responsable: Equipo Docente Cátedra,,,,
fila 5,Los coloreados indican usuarios desactivados,,,,
fila 6,,,,,
fila 7,Usuario,Apellido,Nombre,Nota,Observaciones
fila 8,ALU_1001,Gómez,Manuel,9.00/10.00,aprobada
fila 9,ALU_1004,Flores,Gabriel,9.00/10.00,aprobada
fila 10,ALU_1005,Vega,Agustín,9.00/10.00,aprobada


Archivo,Fila del encabezado,Información útil arriba del encabezado
Padrón de matriculados,2,Código y nombre de la materia
Participación,6 y 7 (encabezado doble),Fecha de exportación y filtro de fechas
Trazabilidad,6,Fecha de exportación y filtro de fechas
Calificaciones,7,Fecha de exportación y nombre de la evaluación


> **Decisión:** leer cada archivo desde la fila de su encabezado. En calificaciones, la cabecera se aprovecha
> para obtener el nombre de la evaluación. Si el LMS cambia el formato, `limpieza.py` se detiene con un error
> en vez de cargar columnas corridas.

## 2. Padrón de matriculados

Cada fila es un **trámite de inscripción**, no un alumno: un mismo alumno puede darse de baja y volver a
inscribirse, y cada movimiento queda como una fila.

In [5]:
matricula_raw = pd.concat(
    [pd.read_excel(f, header=1).assign(materia=nfc(f.stem)) for f in (RAW / "Matriculados").glob("*.xlsx")],
    ignore_index=True,
)
display(matricula_raw.tail(2).fillna(""))
filas_totales = matricula_raw["Identificación"].isna().sum()
matricula_raw = matricula_raw.dropna(subset=["Identificación"])
matricula_raw["fecha"] = pd.to_datetime(matricula_raw["Fecha inscripción"], format="%d/%m/%Y %H:%M:%S")
tramites = matricula_raw.groupby(["materia", "Identificación"]).size()

chequeos([
    ["Fila de totales al final ('Cantidad de filas:')", f"{filas_totales} (una por archivo)",
     "Se quita: no es un alumno"],
    ["Alumnos con un solo trámite", (tramites == 1).sum(), "—"],
    ["Alumnos con 2 o 3 trámites (bajas y reinscripciones)", (tramites > 1).sum(),
     "Se deja una fila por alumno con el estado vigente"],
    ["Bajas sin 'Instancia'", matricula_raw.loc[matricula_raw["Estado Insc."] == "Baja", "Instancia"].isna().sum(),
     "Es esperable: la instancia solo existe si la inscripción está aceptada"],
], "Problemas encontrados en el padrón")

,Alumno,Identificación,Comisión,Estado Insc.,Instancia,Fecha inscripción,Nro de Transacción,materia
514,"DOMÍNGUEZ, NAHUEL",ALU_1147,2319.0,Aceptada,Regularidad,14/08/2025 17:26:38,300816.0,Álgebra
515,Cantidad de filas:,,,,,,,Álgebra


Qué se revisó,Resultado,Qué hacemos
Fila de totales al final ('Cantidad de filas:'),3 (una por archivo),Se quita: no es un alumno
Alumnos con un solo trámite,488,—
Alumnos con 2 o 3 trámites (bajas y reinscripciones),11,Se deja una fila por alumno con el estado vigente
Bajas sin 'Instancia',21,Es esperable: la instancia solo existe si la inscripción está aceptada


**Regla para el estado vigente:** si el alumno tiene algún trámite aceptado, vale el último aceptado; si solo
tiene bajas, queda como baja. La tabla muestra todos los casos con más de un trámite y marca cuál se conserva.

In [6]:
casos = matricula_raw[matricula_raw.groupby(["materia", "Identificación"])["Alumno"].transform("size") > 1].copy()
casos["aceptada"] = casos["Estado Insc."].eq("Aceptada")
vigente = casos.sort_values(["aceptada", "fecha"]).groupby(["materia", "Identificación"]).tail(1).index
casos = casos.sort_values(["materia", "Alumno", "fecha"])
tabla(pd.DataFrame({
    "Materia": casos["materia"], "Alumno": casos["Alumno"], "ID": casos["Identificación"],
    "Fecha del trámite": casos["fecha"].dt.strftime("%d/%m/%Y %H:%M"), "Estado": casos["Estado Insc."],
    "¿Se conserva?": ["✔ vigente" if i in vigente else "" for i in casos.index],
}), "Alumnos con más de un trámite de inscripción", indice=False)

Materia,Alumno,ID,Fecha del trámite,Estado,¿Se conserva?
Algoritmos y Estructuras de Datos,"ALVAREZ, BRUNO",ALU_1064,13/08/2025 20:20,Baja,
Algoritmos y Estructuras de Datos,"ALVAREZ, BRUNO",ALU_1064,13/08/2025 20:21,Aceptada,✔ vigente
Algoritmos y Estructuras de Datos,"GARCÍA, LUCAS",ALU_1028,02/08/2025 16:06,Baja,
Algoritmos y Estructuras de Datos,"GARCÍA, LUCAS",ALU_1028,03/08/2025 00:31,Aceptada,✔ vigente
Algoritmos y Estructuras de Datos,"GARCÍA, MARIANA",ALU_1034,14/08/2025 10:42,Baja,
Algoritmos y Estructuras de Datos,"GARCÍA, MARIANA",ALU_1034,14/08/2025 21:48,Aceptada,✔ vigente
Algoritmos y Estructuras de Datos,"LÓPEZ, SEBASTIÁN",ALU_1282,30/07/2025 16:24,Baja,
Algoritmos y Estructuras de Datos,"LÓPEZ, SEBASTIÁN",ALU_1282,30/07/2025 16:25,Aceptada,✔ vigente
Algoritmos y Estructuras de Datos,"MALDONADO, MARCOS",ALU_1260,01/08/2025 13:10,Baja,
Algoritmos y Estructuras de Datos,"MALDONADO, MARCOS",ALU_1260,01/08/2025 13:12,Aceptada,✔ vigente


> **Decisión:** quitar la fila de totales y dejar **una fila por alumno y materia** con el estado vigente.
> No se pierde información: se agrega la columna `cantidad_bajas`.

In [7]:
matricula = limpieza.limpiar_matricula()
verificacion([
    ["Filas (una por alumno y materia)", tramites.size, len(matricula)],
    ["Alumnos repetidos", 0, matricula.duplicated(["materia", "usuario"]).sum()],
    ["Filas sin ID", 0, matricula["usuario"].isna().sum()],
    ["Alumnos con alguna baja registrada", (matricula_raw.groupby(["materia", "Identificación"])["Estado Insc."]
                                            .apply(lambda s: (s == "Baja").any())).sum(),
     (matricula["cantidad_bajas"] > 0).sum()],
], "Verificación de matricula.parquet")
display(matricula.head())

Verificación,Esperado,Obtenido,¿OK?
Filas (una por alumno y materia),499,499,✔
Alumnos repetidos,0,0,✔
Filas sin ID,0,0,✔
Alumnos con alguna baja registrada,14,14,✔


,materia,comision,usuario,alumno,estado,instancia,fecha_inscripcion,nro_transaccion,cantidad_bajas
0,Algoritmos y Estructuras de Datos,2320,ALU_1001,"GÓMEZ, MANUEL",Aceptada,Regularidad,2025-08-15 10:30:13,300847,0
1,Algoritmos y Estructuras de Datos,2320,ALU_1002,"SILVA, DIEGO",Aceptada,Regularidad,2025-09-24 13:13:30,303617,0
2,Algoritmos y Estructuras de Datos,2320,ALU_1004,"FLORES, GABRIEL",Aceptada,Regularidad,2025-07-31 11:31:32,298536,0
3,Algoritmos y Estructuras de Datos,2320,ALU_1006,"RAMOS, PATRICIO",Aceptada,Regularidad,2025-08-04 22:01:04,299615,0
4,Algoritmos y Estructuras de Datos,2320,ALU_1008,"RAMÍREZ, GABRIEL",Aceptada,Regularidad,2025-08-06 18:44:24,300209,0


## 3. Reporte de participación

Una fila por usuario del aula con su resumen de actividad al momento de exportar (24/10/2025).

### 3.1 Encabezado en dos filas

La fila de arriba agrupa (`Foros`, `Wikis`...) y la de abajo detalla (`Cantidad`, `Cargados`...). Leído tal cual,
quedan columnas sin nombre o con nombres repetidos. La tabla muestra cómo se combinan en un único nombre.

In [8]:
archivo = next(carpeta("Álgebra").glob("reporte_participacion_*.xlsx"))
encabezado = pd.read_excel(archivo, header=None, skiprows=5, nrows=2).T
encabezado.columns = ["Fila 1 del Excel", "Fila 2 del Excel"]
encabezado["Fila 1 del Excel"] = encabezado["Fila 1 del Excel"].ffill()
encabezado = encabezado.fillna("")
encabezado["Nombre de columna que usamos"] = limpieza.COLUMNAS_PARTICIPACION
tabla(encabezado, "Encabezado en dos filas → nombre único", indice=False)

Fila 1 del Excel,Fila 2 del Excel,Nombre de columna que usamos
Usuario,,usuario
Apellido,,apellido
Nombre,,nombre
Email,,email
Aula,,aula
Perfil,,perfil
Fecha alta,,fecha_alta
Avance,,avance
Interacciones,,interacciones
Primera interacción,,primera_interaccion


### 3.2 Valores y tipos de dato

In [9]:
def leer_participacion(materia):
    df = pd.read_excel(next(carpeta(materia).glob("reporte_participacion_*.xlsx")), header=None, skiprows=7)
    df.columns = limpieza.COLUMNAS_PARTICIPACION
    return df.assign(materia=materia)


participacion_raw = pd.concat([leer_participacion(m) for m in MATERIAS], ignore_index=True)
no_aplica = participacion_raw["avance"] == "no aplica"
sin_primera = participacion_raw["primera_interaccion"].isna()

chequeos([
    ["Avance = 'no aplica'",
     f"{no_aplica.sum()} usuarios: {participacion_raw.loc[no_aplica, 'perfil'].value_counts().to_dict()}",
     "Se deja vacío: el avance no aplica al staff"],
    ["Email con dato", participacion_raw["email"].notna().sum(), "Se quita la columna: está vacía"],
    ["Aula", f"{participacion_raw.groupby('materia')['aula'].nunique().max()} valor por materia",
     "Se quita: es la materia"],
    ["Primera/última interacción vacías", sin_primera.sum(), "Se dejan vacías (ver abajo)"],
    ["...de esos, con interacciones > 0", (participacion_raw.loc[sin_primera, "interacciones"] > 0).sum(),
     "No hay inconsistencia: son usuarios que nunca entraron"],
    ["Fechas", "texto dd/mm/aaaa hh:mm:ss", "Se convierten a fecha"],
    ["Tiempo estimado de uso", "texto HH:MM, máximo " + participacion_raw["tiempo_uso"].max(),
     "Se convierte a minutos (ver abajo)"],
], "Problemas encontrados en el reporte de participación")

Qué se revisó,Resultado,Qué hacemos
Avance = 'no aplica',"29 usuarios: {'Invitados': 15, 'Tutores': 11, 'Coordinadores': 3}",Se deja vacío: el avance no aplica al staff
Email con dato,0,Se quita la columna: está vacía
Aula,1 valor por materia,Se quita: es la materia
Primera/última interacción vacías,156,Se dejan vacías (ver abajo)
"...de esos, con interacciones > 0",0,No hay inconsistencia: son usuarios que nunca entraron
Fechas,texto dd/mm/aaaa hh:mm:ss,Se convierten a fecha
Tiempo estimado de uso,"texto HH:MM, máximo 87:47",Se convierte a minutos (ver abajo)


### 3.3 ¿`HH:MM` son horas:minutos o minutos:segundos?

Es la decisión menos obvia. Se interpreta como **horas:minutos** por dos motivos:

- Hay valores como `28:54`. Como minutos:segundos sería posible, pero implicaría que el alumno más activo
  estuvo menos de media hora en toda la cursada.
- El tiempo crece junto con las interacciones (correlación de Spearman cercana a 1). Más de mil interacciones
  en 18 minutos (una interacción por segundo durante toda la cursada) no es creíble; en 18 horas, sí.

In [10]:
alumnos_lms = participacion_raw[participacion_raw["perfil"] == "Alumnos"].copy()
alumnos_lms["minutos"] = alumnos_lms["tiempo_uso"].map(limpieza._tiempo_a_minutos)
top = alumnos_lms.nlargest(5, "minutos")
tabla(pd.DataFrame({
    "Materia": top["materia"], "Alumno": top["usuario"], "Interacciones": top["interacciones"],
    "Tiempo en el Excel": top["tiempo_uso"], "Si fueran horas": top["minutos"] / 60,
    "Si fueran minutos": top["minutos"] / 60,
}).assign(**{"Si fueran horas": lambda d: d["Si fueran horas"].map(lambda h: f"{h:.1f} horas".replace(".", ",")),
             "Si fueran minutos": lambda d: d["Si fueran minutos"].map(lambda m: f"{m:.1f} minutos".replace(".", ","))}),
    "Los 5 alumnos con más tiempo registrado", indice=False)
correlacion = alumnos_lms["interacciones"].rank().corr(alumnos_lms["minutos"].rank())
print(f"Correlación de Spearman entre interacciones y tiempo: {correlacion:.2f}".replace(".", ","))

Materia,Alumno,Interacciones,Tiempo en el Excel,Si fueran horas,Si fueran minutos
Probabilidad y Estadística,ALU_1206,1.163,28:54,"28,9 horas","28,9 minutos"
Probabilidad y Estadística,ALU_1018,917,25:19,"25,3 horas","25,3 minutos"
Probabilidad y Estadística,ALU_1002,1.067,19:17,"19,3 horas","19,3 minutos"
Probabilidad y Estadística,ALU_1115,662,18:44,"18,7 horas","18,7 minutos"
Probabilidad y Estadística,ALU_1004,1.080,18:02,"18,0 horas","18,0 minutos"


Correlación de Spearman entre interacciones y tiempo: 0,96


> **Decisión:** aplanar el encabezado, convertir fechas y tiempo (a minutos, en `tiempo_uso_min`), pasar
> `avance` a número (vacío para el staff) y quitar `email` y `aula`. Se conservan los usuarios de staff con su
> `perfil`, para que cada vista filtre lo que necesite.

In [11]:
participacion = limpieza.limpiar_participacion()
verificacion([
    ["Filas (una por usuario y materia)", len(participacion_raw), len(participacion)],
    ["Usuarios repetidos", 0, participacion.duplicated(["materia", "usuario"]).sum()],
    ["Columnas sin nombre o repetidas", 0, participacion.columns.duplicated().sum()],
    ["Avance vacío solo en staff", True, bool(participacion.loc[participacion["avance"].isna(), "perfil"].ne("Alumnos").all())],
    ["Avance entre 0 y 100", True, bool(participacion["avance"].dropna().between(0, 100).all())],
    ["Tiempo de uso sin convertir", 0, participacion["tiempo_uso_min"].isna().sum()],
], "Verificación de participacion.parquet")
tabla(pd.DataFrame({"Columna": participacion.columns, "Tipo": participacion.dtypes.astype(str).values,
                    "Ejemplo": participacion.iloc[0].astype(str).values}).head(12),
      "Tipos de dato después de la limpieza (primeras columnas)", indice=False)

Verificación,Esperado,Obtenido,¿OK?
Filas (una por usuario y materia),986,986,✔
Usuarios repetidos,0,0,✔
Columnas sin nombre o repetidas,0,0,✔
Avance vacío solo en staff,True,True,✔
Avance entre 0 y 100,True,True,✔
Tiempo de uso sin convertir,0,0,✔


Columna,Tipo,Ejemplo
materia,str,Álgebra
usuario,str,ALU_1001
apellido,str,Gómez
nombre,str,Manuel
perfil,str,Alumnos
fecha_alta,datetime64[us],2025-08-05 00:00:00
avance,Float64,86.0
interacciones,int64,1015
primera_interaccion,datetime64[us],2025-08-05 10:50:31
ultima_interaccion,datetime64[us],2025-10-23 21:47:55


## 4. Reporte de trazabilidad

Una fila por cada acción en el aula: entrar a una sección, abrir un texto, rendir una evaluación, etc.
Es la fuente más grande (unas 160 mil filas) y la que más problemas tiene.

In [12]:
trazabilidad_raw = pd.concat(
    [pd.read_excel(next(carpeta(m).glob("reporte_trazabilidad_*.xlsx")), header=5).assign(materia=m)
     for m in MATERIAS],
    ignore_index=True,
)
texto = trazabilidad_raw["Descripción"].astype(str) + " " + trazabilidad_raw["Acción"].astype(str)
chequeos([
    ["Filas", len(trazabilidad_raw), "—"],
    ["Filas repetidas exactas", trazabilidad_raw.duplicated().sum(), "Se quitan (4.1)"],
    ["Acción vacía", trazabilidad_raw["Acción"].isna().sum(), "Se deduce de la descripción (4.2)"],
    ["Emojis como entidades HTML (&#...;)", f"{texto.str.contains('&#', regex=False).sum()} filas", "Se decodifican (4.3)"],
    ["Acentos escritos como \\x{..}", f"{texto.str.contains('x{', regex=False).sum()} fila", "Se decodifican (4.3)"],
    ["Caracteres de control de Windows-1252", f"{texto.str.contains(chr(0x96), regex=False).sum()} filas",
     "Se convierten al carácter correcto (4.3)"],
    ["Otras columnas vacías", int(trazabilidad_raw.drop(columns="Acción").isna().sum().sum()), "—"],
], "Problemas encontrados en la trazabilidad")

Qué se revisó,Resultado,Qué hacemos
Filas,159804,—
Filas repetidas exactas,686,Se quitan (4.1)
Acción vacía,19,Se deduce de la descripción (4.2)
Emojis como entidades HTML (&#...;),13497 filas,Se decodifican (4.3)
Acentos escritos como \x{..},1 fila,Se decodifican (4.3)
Caracteres de control de Windows-1252,8763 filas,Se convierten al carácter correcto (4.3)
Otras columnas vacías,0,—


### 4.1 Filas repetidas

Son filas idénticas en **todas** las columnas: mismo usuario, misma acción y mismo segundo exacto. El LMS
registró el mismo clic dos veces. Si se dejan, inflan los conteos de interacciones.

In [13]:
dup = trazabilidad_raw[trazabilidad_raw.duplicated(keep=False)]
ejemplo = dup[dup["Usuario"].str.startswith("ALU")].head(4)
tabla(ejemplo[["materia", "Fecha", "Usuario", "Sección", "Acción"]].rename(columns={"materia": "Materia"}),
      "Ejemplo de filas repetidas", indice=False)
tabla(dup.drop_duplicates()["Acción"].value_counts().head(6).rename("Filas repetidas").to_frame(),
      "Acciones con más repeticiones")

Materia,Fecha,Usuario,Sección,Acción
Álgebra,2025-10-10 16:52:51,ALU_1002,Programa,Acceder
Álgebra,2025-10-10 16:52:51,ALU_1002,Programa,Acceder
Álgebra,2025-09-10 20:15:45,ALU_1005,Noticias,Acceder
Álgebra,2025-09-10 20:15:45,ALU_1005,Noticias,Acceder


,Filas repetidas
Acción,
Acceder,256
Ver texto,178
Ver archivo,50
Ver tema,27
Visualizar evaluación,21
Ver calificaciones,17


### 4.2 Acción vacía

La descripción permite saber qué hizo el usuario, así que se completa en lugar de borrar la fila.

In [14]:
vacias = trazabilidad_raw[trazabilidad_raw["Acción"].isna()].rename(columns={"Descripción": "descripcion", "Acción": "accion"})
asignada = vacias.apply(limpieza._completar_accion, axis=1)
tabla(pd.DataFrame({"Comienzo de la descripción": vacias["descripcion"].str[:55] + "...", "Acción asignada": asignada})
      .value_counts().rename("Filas").reset_index(), "Cómo se completa la acción vacía", indice=False)

Comienzo de la descripción,Acción asignada,Filas
Descarga del archivo adjunto a la intervencion del tema...,Descargar adjunto,15
Descarga del archivo '1_1_-_Probabilidad_y_Estadistica....,Descargar,3
Modificacion de intervencion de tema de programas 'Foro...,Editar intervención,1


### 4.3 Texto mal codificado

El LMS exporta emojis como entidades HTML, algunos acentos como `\x{F3}` y algunos signos de puntuación como
caracteres de control. La función `limpiar_texto()` lo corrige; los emojis se quitan porque no aportan información
y complican los filtros.

In [15]:
ejemplos = pd.concat([
    trazabilidad_raw.loc[texto.str.contains("&#", regex=False), "Descripción"].drop_duplicates().head(2),
    trazabilidad_raw.loc[trazabilidad_raw["Acción"].astype(str).str.contains("x{", regex=False), "Acción"].head(1),
    trazabilidad_raw.loc[texto.str.contains(chr(0x96), regex=False) & ~texto.str.contains("&#", regex=False),
                         "Descripción"].head(1),
]).drop_duplicates()
tabla(pd.DataFrame({"Original": ejemplos.str.replace(chr(0x96), "\\x96", regex=False).values,
                    "Limpio": ejemplos.map(limpieza.limpiar_texto).values}),
      "Antes y después de limpiar_texto()", indice=False)

Original,Limpio
Visualización de la noticia '&#128226; Aviso importante \x96 Unificación de comisiones' de la categoría 'Novedades',Visualización de la noticia 'Aviso importante – Unificación de comisiones' de la categoría 'Novedades'
Previsualización de la encuesta '&#128226; Encuesta sobre fecha del 1° parcial ' de la categoría 'Espacio de Encuesta',Previsualización de la encuesta 'Encuesta sobre fecha del 1° parcial ' de la categoría 'Espacio de Encuesta'
Finalizar realizaci\x{F3}n,Finalizar realización
Acceso al texto 'Hoja de Ruta \x96 Unidad 1' de la unidad 'Unidad 1: Vectores',Acceso al texto 'Hoja de Ruta – Unidad 1' de la unidad 'Unidad 1: Vectores'


### 4.4 Columnas que se agregan

La descripción tiene información útil que conviene separar para los tableros: qué recurso se consultó y de qué
unidad. También se agrega la sección principal (`Programa > Textos` → `Programa`) para agrupar.

In [16]:
trazabilidad = limpieza.limpiar_trazabilidad()
muestra = trazabilidad.dropna(subset=["unidad"]).drop_duplicates("accion").head(5)
tabla(muestra[["descripcion", "recurso", "unidad", "seccion_principal"]].rename(columns={
    "descripcion": "Descripción", "recurso": "Recurso", "unidad": "Unidad", "seccion_principal": "Sección principal"}),
    "Columnas extraídas de la descripción", indice=False)

Descripción,Recurso,Unidad,Sección principal
Acceso al texto 'Grabaciones de los encuentros' de la unidad 'Links y Grabaciones',Grabaciones de los encuentros,Links y Grabaciones,Programa
Visualización de tema de programas 'Foro de Consulta – Unidad 1: Introducción al Pensamiento Computacional' de la unidad 'Unidad 1: Introducción al Pensamiento Computaciona',Foro de Consulta – Unidad 1: Introducción al Pensamiento Computacional,Unidad 1: Introducción al Pensamiento Computaciona,Programa
Respuesta al tema de programas 'Primer Foro de la Semana: ¡Nos Presentamos!' de la unidad 'Unidad 1:Introducción al Pensamiento Computacional',Primer Foro de la Semana: ¡Nos Presentamos!,Unidad 1:Introducción al Pensamiento Computacional,Programa
Ver participación del tema de programas 'Primer Foro de la Semana: ¡Nos Presentamos!' de la unidad 'Unidad 1:Introducción al Pensamiento Computacional',Primer Foro de la Semana: ¡Nos Presentamos!,Unidad 1:Introducción al Pensamiento Computacional,Programa
Visualización de la unidad 'Links y Grabaciones',Links y Grabaciones,Links y Grabaciones,Programa


> **Decisión:** quitar las filas repetidas, completar la acción, decodificar el texto y quitar emojis, y
> extraer `recurso`, `unidad` y `seccion_principal`. Se quitan `Apellido`, `Nombre` y `Aula` porque ya están en
> participación; se conservan la IP y los eventos del staff.

In [17]:
cols_texto = ["seccion", "accion", "descripcion"]
restos = sum(trazabilidad[c].str.contains("&#|x\\{", regex=True).sum() for c in cols_texto)
verificacion([
    ["Filas = originales − repetidas", len(trazabilidad_raw) - trazabilidad_raw.duplicated().sum(), len(trazabilidad)],
    ["Filas repetidas", 0, trazabilidad.duplicated().sum()],
    ["Acciones vacías o 'Sin dato'", 0, (trazabilidad["accion"].isna() | trazabilidad["accion"].eq("Sin dato")).sum()],
    ["Restos de codificación (&#, \\x{)", 0, restos],
    ["Fechas sin convertir", 0, trazabilidad["fecha"].isna().sum()],
], "Verificación de trazabilidad.parquet")

Verificación,Esperado,Obtenido,¿OK?
Filas = originales − repetidas,159118,159118,✔
Filas repetidas,0,0,✔
Acciones vacías o 'Sin dato',0,0,✔
"Restos de codificación (&#, \x{)",0,0,✔
Fechas sin convertir,0,0,✔


## 5. Calificaciones

Un archivo por evaluación. El nombre de la evaluación no está en una columna sino en la cabecera, y la nota
viene como texto `obtenida/máxima` (por ejemplo `24.00/25.00`), con escalas distintas según la materia.

In [18]:
calificaciones = limpieza.limpiar_calificaciones()
originales = {f.name: pd.read_excel(f, header=None, nrows=4)[0][2].split(":", 1)[1].strip()
              for m in MATERIAS for f in carpeta(m).glob("calificaciones_alumnos_*.xlsx")}
ejemplo_nota = {f.name: pd.read_excel(f, header=6)["Nota"].iloc[0]
                for m in MATERIAS for f in carpeta(m).glob("calificaciones_alumnos_*.xlsx")}
evaluaciones = (calificaciones.groupby(["materia", "archivo"], sort=False)
                .agg(evaluacion=("evaluacion", "first"), tipo=("tipo_evaluacion", "first"),
                     maxima=("nota_maxima", "first"), alumnos=("usuario", "size")).reset_index())
tabla(pd.DataFrame({
    "Materia": evaluaciones["materia"],
    "Nombre en el Excel": evaluaciones["archivo"].map(originales),
    "Nombre limpio": evaluaciones["evaluacion"],
    "Tipo asignado": evaluaciones["tipo"],
    "Nota en el Excel (ej.)": evaluaciones["archivo"].map(ejemplo_nota),
    "Puntaje máximo": evaluaciones["maxima"],
    "Alumnos": evaluaciones["alumnos"],
}), "Evaluaciones: nombre, tipo y escala", dec=0, indice=False)

Materia,Nombre en el Excel,Nombre limpio,Tipo asignado,Nota en el Excel (ej.),Puntaje máximo,Alumnos
Álgebra,AUTOEVALUACION1,AUTOEVALUACION1,Autoevaluación,9.00/10.00,10,165
Álgebra,AUTOEVALUACION2,AUTOEVALUACION2,Autoevaluación,5.83/10.00,10,152
Álgebra,1° Parcial,1° Parcial,Parcial,24.00/25.00,25,162
Probabilidad y Estadística,&#129504; PARCIAL 1,PARCIAL 1,Parcial,65.00/100.00,100,167
Probabilidad y Estadística,&#128073; Actividad Rápida 1,Actividad Rápida 1,Actividad,100.00/100.00,100,147
Probabilidad y Estadística,&#128073; Actividad Rápida 2,Actividad Rápida 2,Actividad,100.00/100.00,100,137
Probabilidad y Estadística,&#128640;Recuperatorio: Evaluación Semanal Nº1,Recuperatorio: Evaluación Semanal Nº1,Recuperatorio,0.00/100.00,100,21
Probabilidad y Estadística,&#128640;Evaluación Semanal Nº1,Evaluación Semanal Nº1,Evaluación semanal,100.00/100.00,100,136
Probabilidad y Estadística,&#128073; Actividad Rápida 4,Actividad Rápida 4,Actividad,100.00/100.00,100,108
Probabilidad y Estadística,&#128073; Actividad Rápida 3,Actividad Rápida 3,Actividad,100.00/100.00,100,112


In [19]:
notas_raw = pd.concat([pd.read_excel(f, header=6)["Nota"] for m in MATERIAS
                       for f in carpeta(m).glob("calificaciones_alumnos_*.xlsx")])
chequeos([
    ["Notas con formato distinto de 'obtenida/máxima'", (~notas_raw.astype(str).str.match(r"^[\d.]+/[\d.]+$")).sum(),
     "—"],
    ["Notas vacías", notas_raw.isna().sum(), "—"],
    ["Escalas distintas", "10, 25 y 100 puntos", "Se agrega nota_pct (0–100) para poder comparar"],
    ["Alumnos repetidos en una misma evaluación", calificaciones.duplicated(["materia", "evaluacion", "usuario"]).sum(), "—"],
    ["Emojis codificados en el nombre de la evaluación", f"{sum('&#' in v for v in originales.values())} evaluaciones",
     "Se limpian con limpiar_texto()"],
], "Problemas encontrados en las calificaciones")

Qué se revisó,Resultado,Qué hacemos
Notas con formato distinto de 'obtenida/máxima',0,—
Notas vacías,0,—
Escalas distintas,"10, 25 y 100 puntos",Se agrega nota_pct (0–100) para poder comparar
Alumnos repetidos en una misma evaluación,0,—
Emojis codificados en el nombre de la evaluación,9 evaluaciones,Se limpian con limpiar_texto()


### 5.1 ¿Sirve la columna "Observaciones"?

El LMS marca cada nota como `aprobada` o `no aprobada`. Si fuera confiable, todas las aprobadas deberían estar
por encima de cierto umbral y todas las no aprobadas, por debajo.

In [20]:
obs = calificaciones.groupby(["materia", "aprobada"])["nota_pct"].agg(["size", "min", "max"]).reset_index()
obs["aprobada"] = obs["aprobada"].map({True: "aprobada", False: "no aprobada"})
obs.columns = ["Materia", "Observación del LMS", "Notas", "Nota más baja", "Nota más alta"]
tabla(obs, "Rango de notas según la observación del LMS", pct=["Nota más baja", "Nota más alta"], indice=False)

Materia,Observación del LMS,Notas,Nota más baja,Nota más alta
Algoritmos y Estructuras de Datos,no aprobada,1,"50,0 %","50,0 %"
Algoritmos y Estructuras de Datos,aprobada,418,"30,0 %","100,0 %"
Probabilidad y Estadística,no aprobada,79,"0,0 %","75,0 %"
Probabilidad y Estadística,aprobada,816,"65,0 %","100,0 %"
Álgebra,no aprobada,1,"0,0 %","0,0 %"
Álgebra,aprobada,478,"12,0 %","100,0 %"


**No es consistente.** En Álgebra hay notas del 12 % marcadas como aprobadas, y en Algoritmos un 50 % figura
como no aprobado mientras otras notas del 30 % figuran como aprobadas.

> **Decisión:** separar la nota en `nota` y `nota_maxima`, agregar `nota_pct`, limpiar el nombre de la evaluación
> y clasificar su tipo. La columna `aprobada` se conserva, pero **no se usa** para la condición final, que se
> calcula con las reglas de la consigna.

In [21]:
verificacion([
    ["Filas (una por alumno y evaluación)", len(notas_raw), len(calificaciones)],
    ["Notas sin convertir", 0, calificaciones["nota"].isna().sum()],
    ["nota_pct entre 0 y 100", True, bool(calificaciones["nota_pct"].between(0, 100).all())],
    ["Evaluaciones de tipo 'Otra' (sin clasificar)", 0, calificaciones["tipo_evaluacion"].eq("Otra").sum()],
], "Verificación de calificaciones.parquet")

Verificación,Esperado,Obtenido,¿OK?
Filas (una por alumno y evaluación),1793,1793,✔
Notas sin convertir,0,0,✔
nota_pct entre 0 y 100,True,True,✔
Evaluaciones de tipo 'Otra' (sin clasificar),0,0,✔


## 6. Unificación de identificadores

La consigna pide unificar IDs entre fuentes. Se verifica que el usuario (`ALU_xxxx`) sirva como clave común.

In [22]:
patron = r"^(ALU|DOC|STAFF)_\d+$"
nombres = participacion.groupby("usuario")[["apellido", "nombre"]].nunique()
alumnos_aula = set(participacion.loc[participacion["perfil"] == "Alumnos", "usuario"])
chequeos(
    [[f"IDs con formato inválido en {n}", (~s.astype(str).str.match(patron)).sum(), "—"]
     for n, s in [("matrícula", matricula["usuario"]), ("participación", participacion["usuario"]),
                  ("trazabilidad", trazabilidad["usuario"]), ("calificaciones", calificaciones["usuario"])]]
    + [
        ["Mismo ID con distinto nombre entre materias", (nombres > 1).any(axis=1).sum(), "—"],
        ["IDs del padrón que no están en el aula", len(set(matricula["usuario"]) - alumnos_aula), "—"],
        ["IDs con notas que no están en el aula", len(set(calificaciones["usuario"]) - alumnos_aula), "—"],
        ["Alumnos del aula que no están en el padrón", len(alumnos_aula - set(matricula["usuario"])),
         "No se borran: son la alerta de 'no inscriptos' del tablero 7"],
    ],
    "Consistencia de los identificadores entre fuentes",
)

Qué se revisó,Resultado,Qué hacemos
IDs con formato inválido en matrícula,0,—
IDs con formato inválido en participación,0,—
IDs con formato inválido en trazabilidad,0,—
IDs con formato inválido en calificaciones,0,—
Mismo ID con distinto nombre entre materias,0,—
IDs del padrón que no están en el aula,0,—
IDs con notas que no están en el aula,0,—
Alumnos del aula que no están en el padrón,151,No se borran: son la alerta de 'no inscriptos' del tablero 7


> **Decisión:** el usuario es la clave común y no necesita transformación: tiene el mismo formato y el mismo
> nombre en las cuatro fuentes. Los alumnos que están en el aula pero no en el padrón **se conservan**.

## 7. Tratamiento de nulos

Después de limpiar, los únicos vacíos que quedan tienen un significado y se dejan a propósito.

In [23]:
significado = {
    ("matricula", "instancia"): "Alumnos dados de baja: no tienen instancia",
    ("participacion", "avance"): "Staff (tutores, coordinadores, invitados): el avance no aplica",
    ("participacion", "primera_interaccion"): "Usuarios que nunca entraron al aula",
    ("participacion", "ultima_interaccion"): "Usuarios que nunca entraron al aula",
    ("trazabilidad", "recurso"): "Acciones sin un recurso puntual (por ejemplo, ver mensajes recibidos)",
    ("trazabilidad", "unidad"): "Acciones fuera de una unidad del programa (inicio, noticias, foros...)",
}
filas = []
for nombre, df in [("matricula", matricula), ("participacion", participacion),
                   ("trazabilidad", trazabilidad), ("calificaciones", calificaciones)]:
    for col, n in df.isna().sum().items():
        if n:
            filas.append({"Tabla": nombre, "Columna": col, "Vacíos": n, "% de la tabla": n / len(df) * 100,
                          "Qué significa": significado.get((nombre, col), "⚠ revisar")})
tabla(pd.DataFrame(filas), "Valores vacíos en las tablas limpias", pct=["% de la tabla"], indice=False)

Tabla,Columna,Vacíos,% de la tabla,Qué significa
matricula,instancia,7,"1,4 %",Alumnos dados de baja: no tienen instancia
participacion,avance,29,"2,9 %","Staff (tutores, coordinadores, invitados): el avance no aplica"
participacion,primera_interaccion,156,"15,8 %",Usuarios que nunca entraron al aula
participacion,ultima_interaccion,156,"15,8 %",Usuarios que nunca entraron al aula
trazabilidad,recurso,3.916,"2,5 %","Acciones sin un recurso puntual (por ejemplo, ver mensajes recibidos)"
trazabilidad,unidad,79.970,"50,3 %","Acciones fuera de una unidad del programa (inicio, noticias, foros...)"


## 8. Resumen de decisiones de limpieza

| Fuente | Problema | Decisión | Justificación |
|---|---|---|---|
| Todas | Título, filtros y fechas antes del encabezado | Leer desde la fila del encabezado | El formato es fijo; si cambia, el script avisa |
| Matrícula | Fila de totales al final | Quitarla | No es un alumno |
| Matrícula | Varios trámites por alumno | Una fila por alumno con el estado vigente + `cantidad_bajas` | El análisis es por alumno; las bajas se conservan como dato |
| Participación | Encabezado en dos filas | Nombres de columna únicos | Evita columnas sin nombre o repetidas |
| Participación | `tiempo_uso` como texto `HH:MM` | Minutos (`tiempo_uso_min`) | Es horas:minutos: valores de más de 24 y coherencia con las interacciones |
| Participación | `avance = "no aplica"` | Vacío | Solo ocurre en el staff |
| Participación | `email` vacío, `aula` constante | Se quitan | No aportan información |
| Trazabilidad | Filas repetidas exactas | Se quitan | Mismo clic registrado dos veces; inflaría los conteos |
| Trazabilidad | `Acción` vacía | Se deduce de la descripción | La descripción dice qué hizo el usuario; no se pierden eventos |
| Trazabilidad | Texto mal codificado y emojis | `limpiar_texto()` | Textos legibles y filtros que funcionan |
| Trazabilidad | Recurso y unidad dentro de la descripción | Columnas `recurso`, `unidad`, `seccion_principal` | Necesarias para analizar qué contenidos se consultan |
| Calificaciones | Nota como texto `x/y` con escalas distintas | `nota`, `nota_maxima`, `nota_pct` | Permite comparar entre evaluaciones y materias |
| Calificaciones | Nombre de evaluación en la cabecera, con emojis | Columnas `evaluacion` y `tipo_evaluacion` | Permite agrupar y aplicar el esquema de cada materia |
| Calificaciones | `Observaciones` inconsistente | Se conserva, pero no se usa para la condición final | Contradice las notas |
| Todas | IDs | Sin cambios; se conservan los no inscriptos | Son consistentes; los no inscriptos son una alerta de la consigna |